# Fetch IAU constellations

Prototype notebook. No helper functions: rerun cells, inspect intermediates, tweak parsing.

Sources:
- D3 Celestial constellation metadata + boundary polygons
- Stellarium `modern_iau` skyculture lines using HIP ids
- HYG database for brightest star per constellation

Output: one JSON per constellation in `public/data/constellations/`.

In [1]:
import json
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

In [2]:
D3_CONSTELLATIONS_URL = "https://raw.githubusercontent.com/ofrohn/d3-celestial/master/data/constellations.json"
D3_BOUNDS_URL = "https://raw.githubusercontent.com/ofrohn/d3-celestial/master/data/constellations.bounds.json"
STELLARIUM_MODERN_IAU_URL = "https://raw.githubusercontent.com/Stellarium/stellarium/master/skycultures/modern_iau/index.json"
HYG_URL = "https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv"

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"scripts", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUT_DIR = PROJECT_ROOT / "public" / "data" / "constellations"
SCHEMA_PATH = OUT_DIR / "schema.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Fetch raw JSON

In [3]:
with urlopen(D3_CONSTELLATIONS_URL) as response:
    d3_constellations = json.load(response)

with urlopen(D3_BOUNDS_URL) as response:
    d3_bounds = json.load(response)

with urlopen(STELLARIUM_MODERN_IAU_URL) as response:
    stellarium_modern_iau = json.load(response)

len(d3_constellations["features"]), len(d3_bounds["features"]), len(stellarium_modern_iau["constellations"])

(89, 89, 88)

## Metadata: abbreviation, name, genitive, meaning

In [4]:
metadata_rows = []
for feature in d3_constellations["features"]:
    props = feature["properties"]
    metadata_rows.append({
        "abbreviation": feature["id"],
        "name": props.get("name"),
        "iau_abbreviation": props.get("desig"),
        "genitive": props.get("gen"),
        "english_name": props.get("en"),
        "rank": int(props["rank"]) if str(props.get("rank", "")).isdigit() else props.get("rank"),
    })

metadata_df = pd.DataFrame(metadata_rows).sort_values("abbreviation").reset_index(drop=True)
metadata_df.head()

,abbreviation,name,iau_abbreviation,genitive,english_name,rank
0,And,Andromeda,And,Andromedae,Andromeda,1
1,Ant,Antlia,Ant,Antliae,Air Pump,3
2,Aps,Apus,Aps,Apodis,Bird of Paradise,3
3,Aql,Aquila,Aql,Aquilae,Eagle,1
4,Aqr,Aquarius,Aqr,Aquarii,Aquarius,2


In [5]:
stellarium_rows = []
for item in stellarium_modern_iau["constellations"]:
    abbreviation = item["id"].split()[-1]
    common_name = item.get("common_name", {})
    stellarium_rows.append({
        "abbreviation": abbreviation,
        "stellarium_native_name": common_name.get("native"),
        "english_meaning": common_name.get("english"),
        "stellarium_byname": common_name.get("byname"),
        "line_paths_hip": item.get("lines", []),
    })

stellarium_df = pd.DataFrame(stellarium_rows).sort_values("abbreviation").reset_index(drop=True)
stellarium_df.head()

,abbreviation,stellarium_native_name,english_meaning,stellarium_byname,line_paths_hip
0,And,Andromeda,Andromeda,Chained Maiden,"[[677, 3092, 5447, 9640], [113726, 116631, 116805, 116584], [116631, 2912, 3092], [2912, 5447, 4436, 3881, 5434, 760..."
1,Ant,Antlia,Air Pump,None,"[[53502, 51172, 46515]]"
2,Aps,Apus,Bird of Paradise,None,"[[72370, 81065], [80047, 81852, 81065]]"
3,Aql,Aquila,Eagle,None,"[[98036, 97649, 97278, 95501, 93805, 95501, 93747, 95501, 97804, 99473], [93244, 93747], [93805, 93429], [99473, 964..."
4,Aqr,Aquarius,Water Bearer,None,"[[102618, 106278, 109074, 110395, 110960, 111497, 110960, 110672, 109074], [109139, 106278, 109074, 110003, 112961, ..."


## Boundary corners from D3 Celestial

GeoJSON order is preserved. Coordinates are converted to `{ra_deg, ra_hours, dec_deg}`. `ra_deg_signed` keeps source value; `ra_deg` wraps to `[0, 360)`.

In [25]:
boundary_rows = []
for feature in d3_bounds["features"]:
    abbreviation = feature["id"]
    geometry = feature["geometry"]
    rings = []

    if geometry["type"] == "Polygon":
        polygon_list = geometry["coordinates"]
    elif geometry["type"] == "MultiPolygon":
        polygon_list = []
        for polygon in geometry["coordinates"]:
            for ring in polygon:
                polygon_list.append(ring)
    else:
        polygon_list = []

    for ring in polygon_list:
        corner_ring = []
        for coord in ring:
            ra_signed = float(coord[0])
            ra_wrapped = ra_signed % 360.0
            dec = float(coord[1])
            corner_ring.append({
                "ra_deg": ra_wrapped,
                # "ra_hours": ra_wrapped / 15.0,
                "dec_deg": dec,
                "ra_deg_signed": ra_signed,
            })
        rings.append(corner_ring)

    ring_lengths = [len(ring) for ring in rings]
    primary_index = ring_lengths.index(max(ring_lengths)) if ring_lengths else None
    boundary_rows.append({
        "abbreviation": abbreviation,
        "boundary_type": geometry["type"],
        "boundary_rings": rings,
        "boundary_corners": rings[primary_index] if primary_index is not None else [],
        "boundary_ring_count": len(rings),
        "boundary_corner_count": sum(ring_lengths),
    })

boundary_df = pd.DataFrame(boundary_rows).sort_values("abbreviation").reset_index(drop=True)
boundary_df[["abbreviation", "boundary_type", "boundary_ring_count", "boundary_corner_count"]].head()

,abbreviation,boundary_type,boundary_ring_count,boundary_corner_count
0,And,Polygon,1,38
1,Ant,Polygon,1,14
2,Aps,Polygon,1,11
3,Aql,Polygon,1,25
4,Aqr,Polygon,1,24


## Brightest star from HYG

Uses lowest visual magnitude in HYG `con` group. Includes HIP id when available.

In [7]:
hyg_df = pd.read_csv(HYG_URL)
hyg_df.columns.tolist()[:20], hyg_df.shape

(['id',
  'hip',
  'hd',
  'hr',
  'gl',
  'bf',
  'proper',
  'ra',
  'dec',
  'dist',
  'pmra',
  'pmdec',
  'rv',
  'mag',
  'absmag',
  'spect',
  'ci',
  'x',
  'y',
  'z'],
 (119626, 37))

In [26]:
hyg_with_constellation_df = hyg_df[hyg_df["con"].notna()].copy()
hyg_with_constellation_df["mag_sort"] = pd.to_numeric(hyg_with_constellation_df["mag"], errors="coerce")
hyg_with_constellation_df = hyg_with_constellation_df[hyg_with_constellation_df["mag_sort"].notna()].copy()

brightest_rows = []
for abbreviation, group in hyg_with_constellation_df.groupby("con"):
    row = group.sort_values("mag_sort").iloc[0]
    brightest_rows.append({
        "abbreviation": abbreviation,
        "brightest_star_name": row["proper"] if pd.notna(row.get("proper")) and str(row.get("proper")).strip() else row.get("bf"),
        "brightest_star_bf": row["bf"] if pd.notna(row.get("bf")) and str(row.get("bf")).strip() else None,
        "brightest_star_hip": int(row["hip"]) if pd.notna(row.get("hip")) else None,
        "brightest_star_mag": float(row["mag_sort"]),
        "brightest_star_ra_deg": float(row["ra"]) * 15.0 if pd.notna(row.get("ra")) else None,
        "brightest_star_dec_deg": float(row["dec"]) if pd.notna(row.get("dec")) else None,
    })

brightest_df = pd.DataFrame(brightest_rows).sort_values("abbreviation").reset_index(drop=True)
brightest_df.head()

,abbreviation,brightest_star_name,brightest_star_bf,brightest_star_hip,brightest_star_mag,brightest_star_ra_deg,brightest_star_dec_deg
0,And,Mirach,43Bet And,5447,2.07,17.432910,35.620558
1,Ant,Alp Ant,Alp Ant,51172,4.28,156.787950,-31.067779
2,Aps,Alp Aps,Alp Aps,72370,3.83,221.965515,-79.044751
3,Aql,Altair,53Alp Aql,97649,0.76,297.695820,8.868322
4,Aqr,Sadalsuud,22Bet Aqr,106278,2.90,322.889730,-5.571172


## Merge table

In [27]:
constellations_df = metadata_df.merge(stellarium_df, on="abbreviation", how="left")
constellations_df = constellations_df.merge(boundary_df, on="abbreviation", how="left")
constellations_df = constellations_df.merge(brightest_df, on="abbreviation", how="left")

line_path_counts = []
line_vertex_counts = []
for line_paths in constellations_df["line_paths_hip"]:
    if isinstance(line_paths, list):
        line_path_counts.append(len(line_paths))
        vertex_count = 0
        for path in line_paths:
            vertex_count += len(path)
        line_vertex_counts.append(vertex_count)
    else:
        line_path_counts.append(0)
        line_vertex_counts.append(0)

constellations_df["line_path_count"] = line_path_counts
constellations_df["line_vertex_count"] = line_vertex_counts

constellations_df.shape

(91, 23)

In [28]:
constellations_df[[
    "abbreviation", "name", "genitive", "english_name", "english_meaning", "stellarium_byname",
    "brightest_star_name", "brightest_star_bf", "brightest_star_hip", # "brightest_star_mag",
    "boundary_type", "boundary_ring_count", "boundary_corner_count",
    "line_path_count", "line_vertex_count",
]].sort_values("abbreviation")

,abbreviation,name,genitive,english_name,english_meaning,stellarium_byname,brightest_star_name,brightest_star_bf,brightest_star_hip,boundary_type,boundary_ring_count,boundary_corner_count,line_path_count,line_vertex_count
0,And,Andromeda,Andromedae,Andromeda,Andromeda,Chained Maiden,Mirach,43Bet And,5447,Polygon,1,38,5,21
1,Ant,Antlia,Antliae,Air Pump,Air Pump,None,Alp Ant,Alp Ant,51172,Polygon,1,14,1,3
2,Aps,Apus,Apodis,Bird of Paradise,Bird of Paradise,None,Alp Aps,Alp Aps,72370,Polygon,1,11,2,5
3,Aql,Aquila,Aquilae,Eagle,Eagle,None,Altair,53Alp Aql,97649,Polygon,1,25,4,18
4,Aqr,Aquarius,Aquarii,Aquarius,Water Bearer,None,Sadalsuud,22Bet Aqr,106278,Polygon,1,24,2,23
5,Ara,Ara,Arae,Altar,Altar,None,Alp Ara,Alp Ara,85792,Polygon,1,16,1,9
6,Ari,Aries,Arietis,Ram,Ram,None,Hamal,13Alp Ari,9884,Polygon,1,13,1,4
7,Aur,Auriga,Aurigae,Charioteer,Charioteer,None,Capella,13Alp Aur,24608,Polygon,1,22,2,13
8,Boo,Boötes,Boötis,Herdsman,Herdsman,None,Arcturus,16Alp Boo,69673,Polygon,1,21,3,16
9,CMa,Canis Major,Canis Majoris,Great Dog,Greater Dog,None,Sirius,9Alp CMa,32349,Polygon,1,7,3,15


## Build JSON-ready records

In [34]:
GREEK_LETTER_NAMES = {
    "Alp": "Alpha", "Bet": "Beta", "Gam": "Gamma", "Del": "Delta", "Eps": "Epsilon", "Zet": "Zeta",
    "Eta": "Eta", "The": "Theta", "Iot": "Iota", "Kap": "Kappa", "Lam": "Lambda", "Mu": "Mu",
    "Nu": "Nu", "Xi": "Xi", "Omi": "Omicron", "Pi": "Pi", "Rho": "Rho", "Sig": "Sigma",
    "Tau": "Tau", "Ups": "Upsilon", "Phi": "Phi", "Chi": "Chi", "Psi": "Psi", "Ome": "Omega",
}
GREEK_LETTER_SYMBOLS = {
    "Alp": "α", "Bet": "β", "Gam": "γ", "Del": "δ", "Eps": "ε", "Zet": "ζ",
    "Eta": "η", "The": "θ", "Iot": "ι", "Kap": "κ", "Lam": "λ", "Mu": "μ",
    "Nu": "ν", "Xi": "ξ", "Omi": "ο", "Pi": "π", "Rho": "ρ", "Sig": "σ",
    "Tau": "τ", "Ups": "υ", "Phi": "φ", "Chi": "χ", "Psi": "ψ", "Ome": "ω",
}

records = []
for _, row in constellations_df.sort_values("abbreviation").iterrows():
    line_paths = row["line_paths_hip"] if isinstance(row["line_paths_hip"], list) else []
    line_segments = []
    for path in line_paths:
        for index in range(len(path) - 1):
            line_segments.append([int(path[index]), int(path[index + 1])])

    english_name = row["english_name"] if pd.notna(row.get("english_name")) else None
    english_meaning = row["english_meaning"] if pd.notna(row.get("english_meaning")) else None
    stellarium_byname = row["stellarium_byname"] if pd.notna(row.get("stellarium_byname")) else None

    meaning = None
    if english_meaning and english_meaning.casefold() != str(row["name"]).casefold():
        meaning = english_meaning
    elif stellarium_byname and stellarium_byname.casefold() != str(row["name"]).casefold():
        meaning = stellarium_byname
    elif english_name and english_name.casefold() != str(row["name"]).casefold():
        meaning = english_name
    elif english_meaning:
        meaning = english_meaning
    elif stellarium_byname:
        meaning = stellarium_byname
    elif english_name:
        meaning = english_name

    brightest_star = None
    if pd.notna(row.get("brightest_star_name")) or pd.notna(row.get("brightest_star_hip")):
        star_name = row["brightest_star_name"] if pd.notna(row.get("brightest_star_name")) else None
        star_designation = row["brightest_star_bf"] if pd.notna(row.get("brightest_star_bf")) else None
        display_name = star_name if star_name else star_designation
        display_name_greek = display_name

        if star_designation:
            parts = str(star_designation).split()
            bayer_token = parts[0] if parts else ""
            greek_code = None
            greek_prefix = ""
            greek_suffix = ""
            for code in GREEK_LETTER_NAMES:
                if code in bayer_token:
                    greek_code = code
                    greek_prefix = bayer_token.split(code, 1)[0]
                    greek_suffix = bayer_token.split(code, 1)[1]
                    break
            if greek_code:
                greek_prefix_display = f"{greek_prefix} " if greek_prefix else ""
                display_name = f"{greek_prefix_display}{GREEK_LETTER_NAMES[greek_code]}{greek_suffix} {row['genitive']}"
                display_name_greek = f"{greek_prefix_display}{GREEK_LETTER_SYMBOLS[greek_code]}{greek_suffix} {row['genitive']}"

        brightest_star = {
            "name": star_name,
            "designation": star_designation,
            "display_name": display_name,
            "display_name_greek": display_name_greek,
            "hip": int(row["brightest_star_hip"]) if pd.notna(row.get("brightest_star_hip")) else None,
            # "mag": float(row["brightest_star_mag"]) if pd.notna(row.get("brightest_star_mag")) else None,
            "ra_deg": float(row["brightest_star_ra_deg"]) if pd.notna(row.get("brightest_star_ra_deg")) else None,
            "dec_deg": float(row["brightest_star_dec_deg"]) if pd.notna(row.get("brightest_star_dec_deg")) else None,
        }

    records.append({
        "abbreviation": row["abbreviation"],
        "name": row["name"],
        "genitive": row["genitive"],
        "meaning": meaning,
        "rank": int(row["rank"]) if pd.notna(row.get("rank")) else None,
        "boundary": {
            "type": row["boundary_type"],
            "corners": row["boundary_corners"] if isinstance(row["boundary_corners"], list) else [],
            # "rings": row["boundary_rings"] if isinstance(row["boundary_rings"], list) else [],
        },
        # "lines": {
            "paths_hip": [[int(hip) for hip in path] for path in line_paths],
            # "segments_hip": line_segments,
        # },
        "brightest_star": brightest_star,
        # "sources": {
        #     "metadata": D3_CONSTELLATIONS_URL,
        #     "boundaries": D3_BOUNDS_URL,
        #     "lines": STELLARIUM_MODERN_IAU_URL,
        #     "brightest_star": HYG_URL,
        # },
    })

len(records), records[0].keys()

(91,
 dict_keys(['abbreviation', 'name', 'genitive', 'meaning', 'rank', 'boundary', 'paths_hip', 'brightest_star']))

In [30]:
preview_df = pd.json_normalize(records)
preview_df[[
    "abbreviation", "name", "genitive", "meaning",
    "brightest_star.name", "brightest_star.designation",
    "brightest_star.display_name", "brightest_star.display_name_greek",
    "brightest_star.hip", # "brightest_star.mag",
    "boundary.type", "lines.paths_hip"
]].head(20)

,abbreviation,name,genitive,meaning,brightest_star.name,brightest_star.designation,brightest_star.display_name,brightest_star.display_name_greek,brightest_star.hip,boundary.type,lines.paths_hip
0,And,Andromeda,Andromedae,Chained Maiden,Mirach,43Bet And,43 Beta Andromedae,43 β Andromedae,5447,Polygon,"[[677, 3092, 5447, 9640], [113726, 116631, 116805, 116584], [116631, 2912, 3092], [2912, 5447, 4436, 3881, 5434, 760..."
1,Ant,Antlia,Antliae,Air Pump,Alp Ant,Alp Ant,Alpha Antliae,α Antliae,51172,Polygon,"[[53502, 51172, 46515]]"
2,Aps,Apus,Apodis,Bird of Paradise,Alp Aps,Alp Aps,Alpha Apodis,α Apodis,72370,Polygon,"[[72370, 81065], [80047, 81852, 81065]]"
3,Aql,Aquila,Aquilae,Eagle,Altair,53Alp Aql,53 Alpha Aquilae,53 α Aquilae,97649,Polygon,"[[98036, 97649, 97278, 95501, 93805, 95501, 93747, 95501, 97804, 99473], [93244, 93747], [93805, 93429], [99473, 964..."
4,Aqr,Aquarius,Aquarii,Water Bearer,Sadalsuud,22Bet Aqr,22 Beta Aquarii,22 β Aquarii,106278,Polygon,"[[102618, 106278, 109074, 110395, 110960, 111497, 110960, 110672, 109074], [109139, 106278, 109074, 110003, 112961, ..."
5,Ara,Ara,Arae,Altar,Alp Ara,Alp Ara,Alpha Arae,α Arae,85792,Polygon,"[[85267, 85727, 82363, 83081, 83153, 85792, 88714, 85792, 85258]]"
6,Ari,Aries,Arietis,Ram,Hamal,13Alp Ari,13 Alpha Arietis,13 α Arietis,9884,Polygon,"[[8832, 8903, 9884, 13209]]"
7,Aur,Auriga,Aurigae,Charioteer,Capella,13Alp Aur,13 Alpha Aurigae,13 α Aurigae,24608,Polygon,"[[25428, 23015, 23767, 24608, 28360, 28380, 25428], [23767, 23453, 23416, 24608, 28358, 28360]]"
8,Boo,Boötes,Boötis,Herdsman,Arcturus,16Alp Boo,16 Alpha Boötis,16 α Boötis,69673,Polygon,"[[69673, 72105, 74666, 73555, 71075, 71053, 69673, 67927, 67275], [69673, 71795], [71075, 69732, 70497, 69483, 69732]]"
9,CMa,Canis Major,Canis Majoris,Greater Dog,Sirius,9Alp CMa,9 Alpha Canis Majoris,9 α Canis Majoris,32349,Polygon,"[[30324, 32349, 34444, 33579, 34444, 35904], [30324, 31592, 33152, 33579], [32349, 33347, 34045, 33160, 33347]]"


## Write one JSON file per constellation

In [35]:
for record in records:
    path = OUT_DIR / f"{record['abbreviation'].lower()}.json"
    path.parent.mkdir(exist_ok=True, parents=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(record, file, ensure_ascii=False, indent=2)
        file.write("\n")

written_files = sorted(OUT_DIR.glob("*.json"))
len([path for path in written_files if path.name != "schema.json"]), written_files[:5]

(88,
 [PosixPath('/home/sebl/code/etoile-data/data/constellations/and.json'),
  PosixPath('/home/sebl/code/etoile-data/data/constellations/ant.json'),
  PosixPath('/home/sebl/code/etoile-data/data/constellations/aps.json'),
  PosixPath('/home/sebl/code/etoile-data/data/constellations/aql.json'),
  PosixPath('/home/sebl/code/etoile-data/data/constellations/aqr.json')])

## Optional schema validation

In [ ]:
with SCHEMA_PATH.open("r", encoding="utf-8") as file:
    schema = json.load(file)

try:
    import jsonschema
    for record in records:
        jsonschema.validate(record, schema)
    validation_result = "ok"
except ModuleNotFoundError:
    validation_result = "jsonschema not installed; skipped"

validation_result